# Telco Churn Prediction



## 1. Business Problem
The goal of this project is to predict customers who are likely to churn from a telecom company.

## 2. Import & Config


In [1]:
from statistics import quantiles

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier , GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV

## 3. Load Data

In [2]:
df = pd.read_csv('../data/Telco-Customer-Churn.csv')
df.head()
df.describe()
df.shape
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

## 4. EDA

In [3]:
# Correct type errors
df['TotalCharges']=pd.to_numeric(df['TotalCharges'],errors='coerce')
df['TotalCharges'].dtype


dtype('float64')

In [4]:
# Function to identify categorical, numerical, and cardinal variables in the dataset
def grab_col_names(dataframe, cat_th=10, car_th=20):

    # categorical columns
    cat_cols = [col for col in dataframe.columns
                if dataframe[col].dtype == "str"]

    # numerical but categorical
    num_but_cat = [col for col in dataframe.columns
                   if dataframe[col].nunique() < cat_th and
                   dataframe[col].dtype != "str"]

    # categorical but cardinal
    cat_but_car = [col for col in dataframe.columns
                   if dataframe[col].nunique() > car_th and
                   dataframe[col].dtype == "str"]

    # final categorical columns
    cat_cols = cat_cols + num_but_cat
    cat_cols = [col for col in cat_cols if col not in cat_but_car]

    # numerical columns
    num_cols = [col for col in dataframe.columns
                if dataframe[col].dtype != "str"]

    num_cols = [col for col in num_cols if col not in num_but_cat]

    print(f"Observations: {dataframe.shape[0]}")
    print(f"Variables: {dataframe.shape[1]}")
    print(f"cat_cols: {len(cat_cols)}")
    print(f"num_cols: {len(num_cols)}")
    print(f"cat_but_car: {len(cat_but_car)}")

    return cat_cols, num_cols, cat_but_car
cat_cols, num_cols, cat_but_car = grab_col_names(df)


Observations: 7043
Variables: 21
cat_cols: 17
num_cols: 3
cat_but_car: 1


In [5]:
# Categorical Variable Analysis
def cat_summary(dataframe, col_name):
    print(pd.DataFrame({
        col_name: dataframe[col_name].value_counts(),
        "Ratio":100*dataframe[col_name].value_counts()/len(dataframe)
    }))
    print("##########################################")

for col in cat_cols:
    cat_summary(df,col)


        gender     Ratio
gender                  
Male      3555  50.47565
Female    3488  49.52435
##########################################
         Partner     Ratio
Partner                   
No          3641  51.69672
Yes         3402  48.30328
##########################################
            Dependents      Ratio
Dependents                       
No                4933  70.041176
Yes               2110  29.958824
##########################################
              PhoneService      Ratio
PhoneService                         
Yes                   6361  90.316626
No                     682   9.683374
##########################################
                  MultipleLines      Ratio
MultipleLines                             
No                         3390  48.132898
Yes                        2971  42.183729
No phone service            682   9.683374
##########################################
                 InternetService      Ratio
InternetService               

In [6]:
# Numerical Variable Analysis
def num_summary(dataframe,numerical_cols):
    quantiles= [0.05,0.10,0.20,0.30,0.40,0.50,
                 0.60,0.70,0.80,0.90,0.95,0.99]
    print(dataframe[numerical_cols].describe().T)
for col in num_cols:
    num_summary(df, col)

count    7043.000000
mean       32.371149
std        24.559481
min         0.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: tenure, dtype: float64
count    7043.000000
mean       64.761692
std        30.090047
min        18.250000
25%        35.500000
50%        70.350000
75%        89.850000
max       118.750000
Name: MonthlyCharges, dtype: float64
count    7032.000000
mean     2283.300441
std      2266.771362
min        18.800000
25%       401.450000
50%      1397.475000
75%      3794.737500
max      8684.800000
Name: TotalCharges, dtype: float64


In [7]:
# Categorical Variables and Target Variable Analysis
df["Churn"]=df["Churn"].map({"Yes":1,"No":0})

def target_summary_with_cat(dataframe, target, categorical_col):

    print(pd.DataFrame({
        "TARGET_MEAN": dataframe.groupby(categorical_col)[target].mean()
    }))

for col in cat_cols:
    target_summary_with_cat(df,"Churn",col)

        TARGET_MEAN
gender             
Female     0.269209
Male       0.261603
         TARGET_MEAN
Partner             
No          0.329580
Yes         0.196649
            TARGET_MEAN
Dependents             
No             0.312791
Yes            0.154502
              TARGET_MEAN
PhoneService             
No               0.249267
Yes              0.267096
                  TARGET_MEAN
MultipleLines                
No                   0.250442
No phone service     0.249267
Yes                  0.286099
                 TARGET_MEAN
InternetService             
DSL                 0.189591
Fiber optic         0.418928
No                  0.074050
                     TARGET_MEAN
OnlineSecurity                  
No                      0.417667
No internet service     0.074050
Yes                     0.146112
                     TARGET_MEAN
OnlineBackup                    
No                      0.399288
No internet service     0.074050
Yes                     0.215315
           

In [8]:
# Define lower and upper thresholds for outlier detection using the IQR method
def outlier_thresholds(dataframe, col_name, q1=0.25 , q3=0.75):


    quartile1=dataframe[col_name].quantile(q1)
    quartile3=dataframe[col_name].quantile(q3)

    interquantile_range = quartile3 - quartile1

    up_limit=quartile3 + 1.5*interquantile_range
    low_limit=quartile1 - 1.5*interquantile_range

    return low_limit, up_limit

def check_outlier(dataframe, col_name):

    low_limit, up_limit=outlier_thresholds(dataframe, col_name)

    if dataframe[
        (dataframe[col_name] >up_limit) |
        (dataframe[col_name] < low_limit)
    ].any(axis=None):
        return True
    else:
        return False

for col in num_cols:
    print(col, check_outlier(df, col))

tenure False
MonthlyCharges False
TotalCharges False


## 5. Feature Engineering

In [9]:
# Missing Values
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [10]:
# Missing values were observed in the TotalCharges variable. Since the number of missing observations was very small, related rows were removed from the dataset.
df.dropna(inplace=True)

In [11]:
# Encoding
binary_cols = [col for col in df.columns
               if df[col].dtype == "str" and
               df[col].nunique() == 2]

# Label Encoder
from sklearn.preprocessing import LabelEncoder

def label_encoder(dataframe, binary_col):
    labelencoder=LabelEncoder()
    dataframe[binary_col]=labelencoder.fit_transform(dataframe[binary_col])
    return dataframe

for col in binary_cols:
    df=label_encoder(df,col)

# One Hot Encoding
ohe_cols=[col for col in df.columns
          if 10>=df[col].nunique()>2]
df = pd.get_dummies(df,
                    columns=ohe_cols,
                    drop_first=True)

In [12]:
# Standardization
scaler = StandardScaler()
df[num_cols]=scaler.fit_transform(df[num_cols])

## 6. Modelling

In [13]:
# Split the dataset into training and testing sets
y=df["Churn"]
X=df.drop(["Churn","customerID"], axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)


In [14]:
# This cell defines a set of classification models for comparing their performance in a unified machine learning pipeline.
models=[
    ("LR",LogisticRegression()),
    ("CART",DecisionTreeClassifier()),
    ("KN",KNeighborsClassifier()),
    ("RF",RandomForestClassifier()),
    ("GB",GradientBoostingClassifier()),
    ("XGB",XGBClassifier(eval_metric="logloss")),
    ("LGBM",LGBMClassifier()),
    ("CatBoost",CatBoostClassifier(verbose=False))
]

In [15]:
# This loop evaluates multiple classification models using 5-fold cross-validation and prints their average Accuracy, F1-score, and ROC-AUC for comparison.
for name,model in models:
    cv_results = cross_validate(
        model,
        X,
        y,
        cv=5,
        scoring=["accuracy","roc_auc","f1"]
    )
    print(f"########## {name} ##########")

    print(f"Accuracy:{cv_results['test_accuracy'].mean():.4f}")
    print(f"F1-score:{cv_results['test_f1'].mean():.4f}")
    print(f"ROC-AUC:{cv_results['test_roc_auc'].mean():.4f}")


########## LR ##########
Accuracy:0.8035
F1-score:0.5981
ROC-AUC:0.8452
########## CART ##########
Accuracy:0.7288
F1-score:0.4921
ROC-AUC:0.6545
########## KN ##########
Accuracy:0.7673
F1-score:0.5473
ROC-AUC:0.7816
########## RF ##########
Accuracy:0.7898
F1-score:0.5491
ROC-AUC:0.8233
########## GB ##########
Accuracy:0.8046
F1-score:0.5895
ROC-AUC:0.8455
########## XGB ##########
Accuracy:0.7801
F1-score:0.5513
ROC-AUC:0.8191
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1495, number of negative: 4130
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000265 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 637
[LightGBM] [Info] Number of data points in the train set: 5625, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.265778 -> inits

In [16]:
# Logistic Regression model is tuned using GridSearchCV to find the best hyperparameters and then evaluated using 5-fold cross-validation on multiple metrics (Accuracy, F1-score, ROC-AUC).
lr_model = LogisticRegression()

lr_params = {
    "C": [0.01, 0.1, 1, 10],
    "solver": ["liblinear", "lbfgs"]
}

# GridSearchCV
lr_best_grid = GridSearchCV(
    lr_model,
    lr_params,
    cv=5,
    n_jobs=-1,
    verbose=True
).fit(X_train, y_train)

# Best parameters
print(lr_best_grid.best_params_)


# Final model
lr_final = lr_model.set_params(
    **lr_best_grid.best_params_
).fit(X_train, y_train)


# Cross validation results
cross_validate(
    lr_final,
    X,
    y,
    cv=5,
    scoring=["accuracy","f1","roc_auc"]
)

# Evaluation metrics
print("Accuracy:", cv_results['test_accuracy'].mean())
print("F1:", cv_results['test_f1'].mean())
print("ROC-AUC:", cv_results['test_roc_auc'].mean())


Fitting 5 folds for each of 8 candidates, totalling 40 fits
{'C': 10, 'solver': 'lbfgs'}
Accuracy: 0.7942256811856183
F1: 0.5641592871774034
ROC-AUC: 0.8391014107001865


In [17]:
# Gradient Boosting Classifier is optimized using GridSearchCV and then evaluated with 5-fold cross-validation based on Accuracy, F1-score, and ROC-AUC metrics.

gb_model = GradientBoostingClassifier()

gb_params = {
    "learning_rate": [0.01, 0.1],
    "max_depth": [3,5],
    "n_estimators": [100,200]
}

# GridSearchCV
gb_best_grid = GridSearchCV(
    gb_model,
    gb_params,
    cv=5,
    n_jobs=-1,
    verbose=True
).fit(X_train, y_train)

# Best parameters
print(gb_best_grid.best_params_)

# Final model
gb_final = gb_model.set_params(
    **gb_best_grid.best_params_
).fit(X_train, y_train)

# Cross validation results
cv_results = cross_validate(
    gb_final,
    X,
    y,
    cv=5,
    scoring=["accuracy","f1","roc_auc"]
)

# Evaluation metrics
print("Accuracy:", cv_results['test_accuracy'].mean())
print("F1:", cv_results['test_f1'].mean())
print("ROC-AUC:", cv_results['test_roc_auc'].mean())

Fitting 5 folds for each of 8 candidates, totalling 40 fits
{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}
Accuracy: 0.8046055032700752
F1: 0.5895450158297088
ROC-AUC: 0.845548904659435


In [18]:
# CatBoost model hyperparameters are optimized using GridSearchCV and the best model is evaluated with 5-fold cross-validation using Accuracy, F1-score, and ROC-AUC metrics.
cat_model = CatBoostClassifier(verbose=False)

cat_params = {
    "iterations": [200, 500],
    "learning_rate": [0.01, 0.1],
    "depth": [4, 6]
}

# GridSearchCV
cat_best_grid = GridSearchCV(
    cat_model,
    cat_params,
    cv=5,
    n_jobs=-1,
    verbose=True
).fit(X_train, y_train)

# Best parameters
print(cat_best_grid.best_params_)

# Final model
cat_final = cat_model.set_params(
    **cat_best_grid.best_params_
).fit(X_train, y_train)

# Cross validation results
cv_results = cross_validate(
    cat_final,
    X,
    y,
    cv=5,
    scoring=["accuracy", "f1", "roc_auc"]
)

# Evaluation metrics
print("Accuracy:", cv_results['test_accuracy'].mean())
print("F1:", cv_results['test_f1'].mean())
print("ROC-AUC:", cv_results['test_roc_auc'].mean())

Fitting 5 folds for each of 8 candidates, totalling 40 fits
{'depth': 6, 'iterations': 500, 'learning_rate': 0.01}
Accuracy: 0.8047481551802054
F1: 0.5818581485917095
ROC-AUC: 0.8482914013341437
